# Stage 6 — Generate Final Submission (Revenue + COGS Recursive)

**Datathon 2026 — The Gridbreaker (VinTelligence / VinUniversity)**

- Load model forecasts from Stage 4 (`ensemble_predictions.csv`)
- Select final columns for both Revenue and COGS (ensemble preferred)
- Align by `Date` with `sample_submission.csv` to keep exact order
- Validate format và save `submission.csv`

In [1]:
import pandas as pd
import os
from pathlib import Path

BASE_DIR = Path().resolve()
DATA_DIR = BASE_DIR / 'data_raw'
OUT_DIR = BASE_DIR / 'output'
print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load Recursive Revenue + Sample Submission

In [2]:
# Load recursive/ensemble predictions
pred = pd.read_csv(os.path.join(OUT_DIR, 'ensemble_predictions.csv'), parse_dates=['date'])
print('Predictions:', pred.shape)
print(pred.head(3))

# Load sample submission to keep exact Date order
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'), parse_dates=['Date'])
print('\nSample submission:', sample.shape)
print(sample.head(3))
print('Columns:', sample.columns.tolist())

# Choose final revenue column priority
candidate_rev_cols = ['ensemble_rev', 'lgbm_rev', 'catboost_rev', 'prophet_rev']
available_rev = [c for c in candidate_rev_cols if c in pred.columns]
if not available_rev:
    raise ValueError(f'No revenue prediction columns found. Expected one of: {candidate_rev_cols}')
FINAL_REV_COL = available_rev[0]

# Choose final COGS column priority
candidate_cogs_cols = ['ensemble_cogs', 'lgbm_cogs', 'catboost_cogs', 'prophet_cogs']
available_cogs = [c for c in candidate_cogs_cols if c in pred.columns]
if not available_cogs:
    raise ValueError(f'No COGS prediction columns found. Expected one of: {candidate_cogs_cols}')
FINAL_COGS_COL = available_cogs[0]

print(f'\nUsing final revenue column: {FINAL_REV_COL}')
print(f'Using final COGS column   : {FINAL_COGS_COL}')

Predictions: (548, 9)
        date   prophet_rev      lgbm_rev  catboost_rev  ensemble_rev  \
0 2023-01-01  1.678906e+06  2.435059e+06  2.218154e+06  2.338724e+06   
1 2023-01-02  1.869118e+06  2.077670e+06  2.054561e+06  2.067407e+06   
2 2023-01-03  1.858645e+06  1.904573e+06  1.590833e+06  1.765230e+06   

   prophet_cogs     lgbm_cogs  catboost_cogs  ensemble_cogs  
0  1.491121e+06  2.569462e+06   2.321510e+06   2.569462e+06  
1  1.646537e+06  1.678872e+06   1.262263e+06   1.678872e+06  
2  1.657605e+06  1.072497e+06   8.431730e+05   1.072497e+06  

Sample submission: (548, 3)
        Date     Revenue        COGS
0 2023-01-01  2665507.20  2518885.15
1 2023-01-02  1280007.89  1136463.00
2 2023-01-03  1015899.51   822721.12
Columns: ['Date', 'Revenue', 'COGS']

Using final revenue column: ensemble_rev
Using final COGS column   : ensemble_cogs


## 2. Build Submission DataFrame (Revenue predicted, COGS copied)

In [3]:
# Align by date to guarantee exact row order
pred_map_rev = pred.set_index('date')[FINAL_REV_COL]
pred_map_cogs = pred.set_index('date')[FINAL_COGS_COL]
revenue_aligned = sample['Date'].map(pred_map_rev)
cogs_aligned = sample['Date'].map(pred_map_cogs)

if revenue_aligned.isnull().any():
    missing = int(revenue_aligned.isnull().sum())
    raise ValueError(f'Missing Revenue predictions after date alignment: {missing} rows')
if cogs_aligned.isnull().any():
    missing = int(cogs_aligned.isnull().sum())
    raise ValueError(f'Missing COGS predictions after date alignment: {missing} rows')

submission = pd.DataFrame({
    'Date': sample['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': revenue_aligned.clip(lower=0).round(2),
    'COGS': cogs_aligned.clip(lower=0).round(2),
})

print('Submission shape:', submission.shape)
print(submission.head(5))
print(submission.tail(3))

Submission shape: (548, 3)
         Date     Revenue        COGS
0  2023-01-01  2338724.09  2569462.33
1  2023-01-02  2067406.53  1678871.95
2  2023-01-03  1765229.79  1072497.14
3  2023-01-04  1495077.94  1334390.88
4  2023-01-05  1450110.08  1143796.48
           Date     Revenue        COGS
545  2024-06-29  5462678.51  5329866.29
546  2024-06-30  5825287.58  5334579.45
547  2024-07-01  4950004.26  4893420.86


## 3. Validate

In [4]:
errors = []

# Row count
if len(submission) != 548:
    errors.append(f'Row count: {len(submission)} (expected 548)')

# Date range
dates = pd.to_datetime(submission['Date'])
if str(dates.min().date()) != '2023-01-01':
    errors.append(f'Min date: {dates.min().date()} (expected 2023-01-01)')
if str(dates.max().date()) != '2024-07-01':
    errors.append(f'Max date: {dates.max().date()} (expected 2024-07-01)')

# No NaN
nan_count = submission.isnull().sum().sum()
if nan_count > 0:
    errors.append(f'NaN values: {nan_count}')

# No negatives
neg_rev = (submission['Revenue'] < 0).sum()
neg_cogs = (submission['COGS'] < 0).sum()
if neg_rev > 0:
    errors.append(f'Negative Revenue: {neg_rev} rows')
if neg_cogs > 0:
    errors.append(f'Negative COGS: {neg_cogs} rows')

# Columns match
if list(submission.columns) != ['Date', 'Revenue', 'COGS']:
    errors.append(f'Columns mismatch: {submission.columns.tolist()}')

# Date alignment with sample
sample_dates = pd.to_datetime(sample['Date']).dt.strftime('%Y-%m-%d').values
sub_dates = submission['Date'].values
mismatch = (sample_dates != sub_dates).sum()
if mismatch > 0:
    errors.append(f'Date mismatch with sample_submission: {mismatch} rows')

if errors:
    for e in errors:
        print('❌', e)
else:
    print('✅ All validation checks passed!')
    print(f'   Rows      : {len(submission)}')
    print(f'   Date range: {submission["Date"].iloc[0]} → {submission["Date"].iloc[-1]}')
    print(f'   Revenue   : min={submission["Revenue"].min():,.2f}  max={submission["Revenue"].max():,.2f}  mean={submission["Revenue"].mean():,.2f}')
    print(f'   COGS      : min={submission["COGS"].min():,.2f}  max={submission["COGS"].max():,.2f}  mean={submission["COGS"].mean():,.2f}')

✅ All validation checks passed!
   Rows      : 548
   Date range: 2023-01-01 → 2024-07-01
   Revenue   : min=1,306,261.40  max=9,521,596.44  mean=3,629,048.26
   COGS      : min=975,777.73  max=8,255,473.96  mean=3,037,000.54


## 4. Save submission.csv

In [5]:
out_path = os.path.join(OUT_DIR, 'submission.csv')
submission.to_csv(out_path, index=False)
print(f'submission.csv saved → {out_path}')

# Verify by re-reading
check = pd.read_csv(out_path)
print(f'Re-read shape: {check.shape}')
print(check.head(3))

submission.csv saved → D:\push_code\Phan 3 Mo hinh Du bao Doanh thu\output\submission.csv
Re-read shape: (548, 3)
         Date     Revenue        COGS
0  2023-01-01  2338724.09  2569462.33
1  2023-01-02  2067406.53  1678871.95
2  2023-01-03  1765229.79  1072497.14


## 5. Summary

In [6]:
print('=' * 60)
print('STAGE 6 COMPLETE — Final Submission')
print('=' * 60)
print(f'  File    : submission.csv')
print(f'  Rows    : {len(submission)}')
print(f'  Dates   : {submission["Date"].iloc[0]} -> {submission["Date"].iloc[-1]}')
print(f'  Revenue : mean = {submission["Revenue"].mean():>15,.2f} VND/day')
print(f'  COGS    : mean = {submission["COGS"].mean():>15,.2f} VND/day')
print()
print('Pipeline summary (updated):')
print('  Stage 3 ✓  Revenue + COGS training + recursive forecasting')
print('  Stage 4 ✓  Revenue + COGS ensemble blending')
print('  Stage 6 ✓  Revenue + COGS from model predictions')
print()
print(f'Final revenue source column: {FINAL_REV_COL}')
print(f'Final COGS source column   : {FINAL_COGS_COL}')

STAGE 6 COMPLETE — Final Submission
  File    : submission.csv
  Rows    : 548
  Dates   : 2023-01-01 -> 2024-07-01
  Revenue : mean =    3,629,048.26 VND/day
  COGS    : mean =    3,037,000.54 VND/day

Pipeline summary (updated):
  Stage 3 ✓  Revenue + COGS training + recursive forecasting
  Stage 4 ✓  Revenue + COGS ensemble blending
  Stage 6 ✓  Revenue + COGS from model predictions

Final revenue source column: ensemble_rev
Final COGS source column   : ensemble_cogs
